# Organize data

## Pair videos and frames

In [2]:
import csv

# csv events
def get_events_for_video(video_id):
    events = []
    with open('key_frames.csv', 'r') as csvfile:
        reader = csv.reader(csvfile)
        next(reader)
        for row in reader:
            if int(row[0]) == video_id:
                events.append(int(row[1]))
    return events

### label key frames in dataset

In [3]:

import math
from collections import defaultdict
from moviepy.editor import VideoFileClip


def label_frames(video_dir, num_classes=8):
    frame_labels = defaultdict(list)

    for video_id_dir in os.listdir(video_dir):

        # video_frames_dir = os.path.join(frames_dir, video_id_dir)
        video_id = int(video_id_dir.split(".")[0])

        # video_entry = next((video for video in golfdb if video[0] == video_id), None)
        # if video_entry is None:
        #     print(f"video ID {video_id} not found")
        #     continue

        key_frames = get_events_for_video(video_id)

        video_path = os.path.join(video_dir, f'{video_id}.mp4')
        # print(video_path)
        # print(f"Loading video from path: {video_path}")
        clip = VideoFileClip(video_path)

        # print(f"Video {video_id} FPS: {clip.fps}, Duration: {clip.duration}")


        fps = math.ceil(clip.fps)
        total_frames = int(clip.duration * fps)
        # print(total_frames)

        # print(f"video {video_path} has {total_frames} total frames")

        key_frame_dict = {int(kf): i for i, kf in enumerate(key_frames)}

        for frame_num in range(total_frames):
            if frame_num in key_frame_dict:
                event_index = key_frame_dict[frame_num]
            else:
                event_index = num_classes

            frame_labels[video_id].append((frame_num, event_index))

    return frame_labels

# frames_dir = "../frames_160"
# video_dir = "../videos_160"
# frame_labels = label_frames(video_dir)
# print(len(frame_labels))

### todo: oversample key events

### inits

In [4]:
from torch.utils.data import DataLoader
from torchvision import transforms
import pickle

# Define transformations to preprocess video frames for MobileNetV2
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# frame_labels = label_frames(video_dir)

# with open('golf_frame_labels.pkl', 'wb') as f:
#     pickle.dump(frame_labels, f)

# get labels for frames
with open('golf_frame_labels.pkl', 'rb') as f:
    frame_labels = pickle.load(f)

In [5]:
print(frame_labels)

defaultdict(<class 'list'>, {1013: [(0, 8), (1, 8), (2, 8), (3, 8), (4, 8), (5, 8), (6, 8), (7, 8), (8, 8), (9, 8), (10, 8), (11, 8), (12, 8), (13, 8), (14, 8), (15, 8), (16, 8), (17, 8), (18, 8), (19, 8), (20, 8), (21, 8), (22, 8), (23, 8), (24, 8), (25, 8), (26, 8), (27, 8), (28, 8), (29, 8), (30, 8), (31, 8), (32, 8), (33, 8), (34, 8), (35, 8), (36, 8), (37, 8), (38, 8), (39, 8), (40, 8), (41, 8), (42, 8), (43, 8), (44, 8), (45, 8), (46, 8), (47, 8), (48, 8), (49, 8), (50, 8), (51, 8), (52, 8), (53, 8), (54, 8), (55, 8), (56, 8), (57, 8), (58, 0), (59, 8), (60, 8), (61, 8), (62, 8), (63, 8), (64, 8), (65, 8), (66, 8), (67, 8), (68, 8), (69, 8), (70, 8), (71, 1), (72, 8), (73, 8), (74, 2), (75, 8), (76, 8), (77, 8), (78, 8), (79, 8), (80, 8), (81, 8), (82, 8), (83, 3), (84, 8), (85, 8), (86, 8), (87, 8), (88, 4), (89, 8), (90, 8), (91, 8), (92, 5), (93, 8), (94, 6), (95, 8), (96, 8), (97, 8), (98, 8), (99, 8), (100, 8), (101, 8), (102, 8), (103, 8), (104, 8), (105, 8), (106, 8), (107

In [6]:
from data_class import GolfSwingDataset, ToTensor, Normalize

In [7]:
dataset = GolfSwingDataset(
    frame_labels=frame_labels,
    vid_frames_dir='vid_frames',
    seq_length=64,
    transform=transforms.Compose([
        ToTensor(),
        Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    train=True
)

data_loader = DataLoader(
    dataset,
    batch_size=6,
    shuffle=True,
    num_workers=2,
    drop_last=False
)


In [8]:
for i, sample in enumerate(data_loader):
    images, labels = sample['images'], sample['labels']
    print(f"Batch {i} - Images shape: {images.shape}, Labels shape: {labels.shape}")
    if i ==3:
        break


Batch 0 - Images shape: torch.Size([6, 64, 3, 160, 160]), Labels shape: torch.Size([6, 64])
Batch 1 - Images shape: torch.Size([6, 64, 3, 160, 160]), Labels shape: torch.Size([6, 64])
Batch 2 - Images shape: torch.Size([6, 64, 3, 160, 160]), Labels shape: torch.Size([6, 64])
Batch 3 - Images shape: torch.Size([6, 64, 3, 160, 160]), Labels shape: torch.Size([6, 64])


# Model

## Training

In [9]:
import random
# Create train test split datasets/loaders

n_cpu = 2
seq_length = 64
train_bs = 3  # LOWER if encountering memory issues
eval_bs = 1  # use batch size 1 for evaluation to process each full sequence


# create a train/test split (80% train, 20% test)
random_seed = 42
video_ids = list(frame_labels.keys())
rng = random.Random(random_seed)
rng.shuffle(video_ids)
train_ratio = 0.8
split_idx = int(len(video_ids) * train_ratio)
train_video_ids = video_ids[:split_idx]
test_video_ids = video_ids[split_idx:]
train_frame_labels = {vid: frame_labels[vid] for vid in train_video_ids}
test_frame_labels = {vid: frame_labels[vid] for vid in test_video_ids}


# TRAINING SET
train_dataset = GolfSwingDataset(
    frame_labels=train_frame_labels,  # frames and labels dictionary for training
    vid_frames_dir='vid_frames',
    seq_length=seq_length,
    transform=transforms.Compose([
        ToTensor(),
        Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    train=True
)

# TRAINING LOADER
train_loader = DataLoader(
    train_dataset,
    batch_size=train_bs,
    shuffle=True,
    num_workers=n_cpu,
    drop_last=True
)

# TRAINING LOADER FOR EVALUATION
train_eval_loader = DataLoader(
    train_dataset,
    batch_size=eval_bs,
    shuffle=False,
    num_workers=n_cpu,
    drop_last=True,
)

# TESTING SET
test_dataset = GolfSwingDataset(
    frame_labels=test_frame_labels,  # frames and labels dictionary for testing
    vid_frames_dir='vid_frames',
    seq_length=seq_length,
    transform=transforms.Compose([
        ToTensor(),
        Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    train=False
)

# TESTING LOADER
test_eval_loader = DataLoader(
    test_dataset,
    batch_size=eval_bs,
    shuffle=False,
    num_workers=n_cpu,
    drop_last=False
)

In [ ]:
# this cell is not necessary to run before moving on as long as a saved model is in the /models directory

import torch
from EventDetectorTransformer import EventDetector
import os


class AverageMeter:
    """Tracks and stores the average and current value."""
    def __init__(self):
        self.reset()

    def reset(self):
        """Resets all values to start fresh."""
        self.val = 0  # Current value
        self.avg = 0  # Average value
        self.sum = 0  # Sum of all values
        self.count = 0  # Number of updates

    def update(self, val, n=1):


        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count


if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f"Using device: {device}")

if __name__ == '__main__':


    split = 1
    iterations = 100
    epochs = 30
    it_save = 100


    model = EventDetector(
    width_mult=1.0,
    num_transformer_layers=2,  # Number of transformer encoder layers
    num_heads=8,               # Number of attention heads in each transformer layer
    transformer_dim=1280,      # Feature dimension (should match the CNN's output dimension)
    dropout=False,
    num_classes=9
    ).to(device)

    # Handle class imbalance: Assign weights to classes
    weights = torch.FloatTensor([1/8, 1/8, 1/8, 1/8, 1/8, 1/8, 1/8, 1/8, 1/35]).to(device)
    criterion = torch.nn.CrossEntropyLoss(weight=weights)

    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


    losses = AverageMeter()


    if not os.path.exists('models'):
        os.mkdir('models')

    # Training loop
    i = 0
    while i < epochs:
        for sample in train_loader:
            images, labels = sample['images'].to(device), sample['labels'].to(device)


            labels = labels.view(train_bs * seq_length)


            logits = model(images)


            loss = criterion(logits, labels)


            optimizer.zero_grad()
            loss.backward()
            optimizer.step()


            losses.update(loss.item(), images.size(0))


            print(f"Iteration: {i}\tLoss: {losses.val:.4f} (Average Loss: {losses.avg:.4f})")


            # i += 1
            #if i % it_save == 0:
            #    torch.save({
            #        'optimizer_state_dict': optimizer.state_dict(),
            #        'model_state_dict': model.state_dict()
            #    }, f'models/latestTransformerModel.pth.tar')


            # if i == iterations:
            #     break
        torch.save({
                'optimizer_state_dict': optimizer.state_dict(),
                'model_state_dict': model.state_dict()
            }, f'models/latestTransformerModel{i}.pth.tar')
        i += 1


Using device: cpu
Iteration: 0	Loss: 2.1524 (Average Loss: 2.1524)


In [10]:
import torch
import torch.nn.functional as F
import numpy as np
from EventDetectorTransformer import EventDetector


def correct_preds(probs, labels, tol=-1):
    """
    Gets correct events in full-length sequence using tolerance based on number of frames
    from address to impact.

    Assumes that for frames with an event (labels < 8):
      - The ground truth event frame = frame index + labels[frame]
      - The predicted offset is obtained via np.argmax(probs[frame, :]),
        so the predicted event frame = frame index + predicted offset.
    """
    # make sure labels is numpy array
    if not isinstance(labels, np.ndarray):
        labels = labels.cpu().numpy() if hasattr(labels, 'cpu') else np.array(labels)

    # indices of key events (labels <8)
    events = np.where(labels < 8)[0]
    if len(events) == 0:
        return np.array([]), np.array([]), np.array([]), tol, np.array([])

    preds = np.zeros(len(events))

    # tolerance calculation; if not provided, set to average number of frames between events
    if tol == -1 and len(events) > 1:
        tol = int(max(np.round((events[-1] - events[0]) / (len(events) - 1)), 1))
    elif tol == -1:
        tol = 1

    # compute predicted key frame by adding the predicted offset to the frame index
    for i, event in enumerate(events):
        pred_offset = np.argmax(probs[event, :])
        preds[i] = event + pred_offset

    # ground truth key frame is the frame index + true offset
    gt_times = events + labels[events]
    deltas = np.abs(gt_times - preds)
    correct = (deltas <= tol).astype(np.uint8)

    return events, preds, deltas, tol, correct

In [11]:
def load_model(model, model_path):
    checkpoint = torch.load(model_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()
    return model

In [16]:
def eval(model, seq_length, data_loader, disp):
    all_correct = []
    all_mae = []  # list to store MAE for each sequence

    # for each full sequence, compute the average correct rate
    for i, sample in enumerate(data_loader):
        images, labels = sample['images'], sample['labels']
        images = images.squeeze(0)
        labels = labels.squeeze(0)

        # convert labels to numpy array
        labels_np = labels.cpu().numpy() if torch.is_tensor(labels) else np.array(labels)

        # process the sequence in segments of length seq_length
        probs_list = []
        num_frames = images.shape[0]
        batch = 0
        while batch * seq_length < num_frames:
            if (batch + 1) * seq_length > num_frames:
                image_batch = images[batch * seq_length:]
            else:
                image_batch = images[batch * seq_length:(batch + 1) * seq_length]
            # add batch dimension
            image_batch = image_batch.unsqueeze(0)
            logits = model(image_batch.to(device))
            # detach before converting to numpy
            probs_batch = F.softmax(logits, dim=1).detach().cpu().numpy()  # shape: [segment_frames, num_classes]
            # append the entire array instead of just the first row
            probs_list.append(probs_batch)
            batch += 1

        # concatenate probabilities over all segments into one array of shape [total_frames, num_classes]
        probs = np.concatenate(probs_list, axis=0)
        # evaluate predictions for this sample
        _, _, deltas, _, corr = correct_preds(probs, labels_np)
        if disp:
            print(f"Sample {i}: Correct predictions: {corr}")
        # save the average correct rate for this sequence (if no events, 0 is used)
        all_correct.append(np.mean(corr) if corr.size > 0 else 0)
        # calculate and save MAE for this sequence (if no events, 0 is used)
        all_mae.append(np.mean(deltas) if deltas.size > 0 else 0)

    # compute the overall percentage of correct events (PCE)
    PCE = np.mean(all_correct) if all_correct else 0
    overall_mae = np.mean(all_mae) if all_mae else 0
    return PCE, overall_mae


if __name__ == '__main__':
    seq_length = 64
    n_cpu = 2
    vid_frames_dir = 'vid_frames'
    frame_labels_file = 'golf_frame_labels.pkl'
    model_checkpoint = 'models/latestTransformerModel27.pth.tar'

    model = EventDetector(
        width_mult=1.0,
        num_transformer_layers=2,  # number of transformer encoder layers
        num_heads=8,               # number of attention heads in each transformer layer
        transformer_dim=1280,      # feature dimension (should match the CNN's output dimension)
        dropout=False,
        num_classes=9
    )

    if torch.cuda.is_available():
        device = torch.device('cuda')
    elif torch.backends.mps.is_available():
        device = torch.device('mps')
    else:
        device = torch.device('cpu')

    model = load_model(model, model_checkpoint)

    # load frame labels from pickle file
    import pickle
    with open(frame_labels_file, 'rb') as f:
        frame_labels = pickle.load(f)

    train_PCE, train_overall_mae = eval(model, seq_length, train_eval_loader, disp=True)
    test_PCE, test_overall_mae = eval(model, seq_length, test_eval_loader, disp=True)
    print(f'\nAverage Train PCE: {train_PCE}')
    print(f'Overall Train MAE: {train_overall_mae}')
    print(f'\nAverage Test PCE: {test_PCE}')
    print(f'Overall Test MAE: {test_overall_mae}')


Sample 0: Correct predictions: []
Sample 1: Correct predictions: [1]
Sample 2: Correct predictions: []
Sample 3: Correct predictions: [1]
Sample 4: Correct predictions: []
Sample 5: Correct predictions: []
Sample 6: Correct predictions: []
Sample 7: Correct predictions: [1 1 1 1 1]
Sample 8: Correct predictions: [1 1 1 1 1 1 1]
Sample 9: Correct predictions: []
Sample 10: Correct predictions: []
Sample 11: Correct predictions: [1 1 1 1]
Sample 12: Correct predictions: []
Sample 13: Correct predictions: [1 1 1 1 1]
Sample 14: Correct predictions: [0]
Sample 15: Correct predictions: [1]
Sample 16: Correct predictions: []
Sample 17: Correct predictions: []
Sample 18: Correct predictions: []
Sample 19: Correct predictions: [1 1 1 1 1]
Sample 20: Correct predictions: []
Sample 21: Correct predictions: [0 1 1 1 1 1]
Sample 22: Correct predictions: [1 1 1 1 1 1]
Sample 23: Correct predictions: [0 0 1 1 1 1 1 1]
Sample 24: Correct predictions: [0 0 0 1 1 1 1]
Sample 25: Correct predictions: []

In [17]:
def eval(model, seq_length, data_loader, disp):
    all_correct = []
    all_deltas = []
    videos_processed = 0
    
    for i, sample in enumerate(data_loader):
        images, labels = sample['images'], sample['labels']
        images = images.squeeze(0)
        labels = labels.squeeze(0)
        
        # convert labels to numpy array
        labels_np = labels.cpu().numpy() if torch.is_tensor(labels) else np.array(labels)
        
        # process the sequence in segments of length seq_length
        probs_list = []
        num_frames = images.shape[0]
        batch = 0
        while batch * seq_length < num_frames:
            if (batch + 1) * seq_length > num_frames:
                image_batch = images[batch * seq_length:]
            else:
                image_batch = images[batch * seq_length:(batch + 1) * seq_length]
            # add batch dimension
            image_batch = image_batch.unsqueeze(0)
            logits = model(image_batch.to(device))
            # detach before converting to numpy
            probs_batch = F.softmax(logits, dim=1).detach().cpu().numpy()
            probs_list.append(probs_batch)
            batch += 1
            
        # concatenate probabilities over all segments
        probs = np.concatenate(probs_list, axis=0)
        
        # evaluate predictions for this sample
        events, preds, deltas, tol, corr = correct_preds(probs, labels_np)
        
        # Skip videos with missing events or more than expected
        # if len(events) != 8:
        #     if disp:
        #         print(f"Sample {i}: Warning - Found {len(events)} events instead of expected 8, skipping")
        #     continue
        
        # Count this video as processed
        videos_processed += 1
        
        # Add to overall metrics
        all_correct.extend(corr)
        all_deltas.extend(deltas)
        
        if disp:
            accuracy = np.mean(corr)
            avg_delta = np.mean(deltas)
            print(f"Sample {i}: Accuracy: {accuracy:.4f}, MAE: {avg_delta:.4f}")
    
    # Calculate overall PCE and MAE
    PCE = np.mean(all_correct) if all_correct else 0
    MAE = np.mean(all_deltas) if all_deltas else 0
    
    if disp:
        print(f"\nProcessed {videos_processed} videos with exactly 8 events")
        print(f"Total events evaluated: {len(all_correct)}")
    
    return PCE, MAE, videos_processed


if __name__ == '__main__':
    seq_length = 64
    n_cpu = 2
    vid_frames_dir = 'vid_frames'
    frame_labels_file = 'golf_frame_labels.pkl'
    model_checkpoint = 'models/latestTransformerModel27.pth.tar'

    model = EventDetector(
        width_mult=1.0,
        num_transformer_layers=2,
        num_heads=8,
        transformer_dim=1280,
        dropout=False,
        num_classes=9
    )

    if torch.cuda.is_available():
        device = torch.device('cuda')
    elif torch.backends.mps.is_available():
        device = torch.device('mps')
    else:
        device = torch.device('cpu')

    model = load_model(model, model_checkpoint)

    # load frame labels from pickle file
    import pickle
    with open(frame_labels_file, 'rb') as f:
        frame_labels = pickle.load(f)

    # Run evaluation
    train_PCE, train_MAE, train_videos = eval(model, seq_length, train_eval_loader, disp=True)
    test_PCE, test_MAE, test_videos = eval(model, seq_length, test_eval_loader, disp=True)
    
    print("\n=== RESULTS SUMMARY ===")
    print(f"Train: {train_videos} videos, PCE: {train_PCE:.4f}, MAE: {train_MAE:.4f}")
    print(f"Test: {test_videos} videos, PCE: {test_PCE:.4f}, MAE: {test_MAE:.4f}")

Sample 0: Accuracy: nan, MAE: nan
Sample 1: Accuracy: 1.0000, MAE: 1.0000
Sample 2: Accuracy: nan, MAE: nan
Sample 3: Accuracy: 1.0000, MAE: 7.5000
Sample 4: Accuracy: 0.0000, MAE: 8.0000
Sample 5: Accuracy: nan, MAE: nan
Sample 6: Accuracy: 0.8571, MAE: 4.0000
Sample 7: Accuracy: 0.0000, MAE: 8.0000
Sample 8: Accuracy: 1.0000, MAE: 7.0000
Sample 9: Accuracy: nan, MAE: nan
Sample 10: Accuracy: nan, MAE: nan
Sample 11: Accuracy: 0.0000, MAE: 2.0000
Sample 12: Accuracy: nan, MAE: nan
Sample 13: Accuracy: 1.0000, MAE: 4.0000
Sample 14: Accuracy: 1.0000, MAE: 6.0000
Sample 15: Accuracy: 1.0000, MAE: 3.0000
Sample 16: Accuracy: nan, MAE: nan
Sample 17: Accuracy: 0.8571, MAE: 4.0000
Sample 18: Accuracy: 1.0000, MAE: 5.5000
Sample 19: Accuracy: nan, MAE: nan
Sample 20: Accuracy: nan, MAE: nan
Sample 21: Accuracy: 1.0000, MAE: 4.0000
Sample 22: Accuracy: 1.0000, MAE: 4.0000
Sample 23: Accuracy: 0.5714, MAE: 5.0000
Sample 24: Accuracy: 0.5714, MAE: 5.0000
Sample 25: Accuracy: nan, MAE: nan
Samp